<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_04_1_pytorch_persist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# T81-558: Applications of Deep Neural Networks

**Module 4: PyTorch Training**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).


# Module 4 Material

- **Part 4.1: PyTorch Persistence** [[Video]]() [[Notebook]](t81_558_class_04_1_pytorch_persist.ipynb)
- Part 4.2: Early Stopping and Network Checkpointing [[Video]]() [[Notebook]](t81_558_class_04_2_early_stop.ipynb)
- Part 4.3: K-Fold Cross-validation with PyTorch [[Video]]() [[Notebook]](t81_558_class_04_3_kfold.ipynb)
- Part 4.4: Training Schedules [[Video]]() [[Notebook]](t81_558_class_04_4_schedule.ipynb)
- Part 4.5: Regularization and Dropout [[Video]]() [[Notebook]](t81_558_class_04_5_dropout.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed. We also initialize the PyTorch device to either GPU/MPS (if available) or CPU.


In [1]:
import torch

try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 3.2)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Note: using Google CoLab
Using device: cuda


# Part 4.1: PyTorch Persistence

PyTorch models can be serialized in several formats, each designed for a different stage of the machine learning workflow. Selecting the appropriate format requires understanding whether the intended use is continued training, model sharing, or production deployment. Some formats are self-contained and portable across runtimes and programming languages, while others depend on the availability of the original Python class definition at load time.

The table below describes the major serialization formats a PyTorch practitioner is likely to encounter. The Extension column identifies the file suffix associated with each format. It is worth noting that multiple formats share the `.pth` and `.pt` extensions, meaning the extension alone is not a reliable indicator of a file's contents or structure. The Stores Architecture column indicates whether the format captures sufficient information to reconstruct the model topology, or whether the original class definition must be present in the Python environment at load time. The Safe to Load column reflects whether the format poses a security risk during deserialization. Formats based on Python's pickle mechanism are capable of executing arbitrary code when a file is loaded, which presents a significant risk when loading models from untrusted sources. Newer formats such as SafeTensors and ONNX were designed in part to eliminate this vulnerability. The When to Use column describes the workflow context for which each format is best suited.

Several of the formats in this table are supported natively by Hugging Face Hub, the predominant platform for publishing and distributing pretrained models. SafeTensors is the current Hub standard for full-precision model weights. GGUF has emerged as the standard format for quantized models intended for local inference on consumer hardware. Familiarity with these formats enables practitioners both to publish original models and to make effective use of the large ecosystem of community-contributed models available through the Hub.

| Format | Extension | Stores Architecture? | Safe to Load? | When to Use |
|---|---|---|---|---|
| State Dict | .pth / .pt | No | No | Saving weights during or after training |
| Checkpoint (state dict + optimizer) | .pth / .pt | No | No | Resuming training (weights + optimizer + epoch) |
| Full Model (pickle-based) | .pth / .pkl | No | No | Quick prototyping only — fragile, unsafe to share |
| SafeTensors (Hugging Face standard) | .safetensors | Via config.json | Yes | Sharing models on Hugging Face Hub |
| TorchScript | .pt | Yes | Yes | Production inference; C++ and mobile deployment |
| ONNX | .onnx | Yes | Yes | Cross-platform runtimes (Unity Sentis, TensorRT, mobile) |
| GGUF | .gguf | Yes | Yes | Quantized LLM inference (Ollama, LM Studio, llama.cpp) |


## Saving and Loading a Model with State Dict

The most common format for persisting a PyTorch model during and after training
is the state dict, as shown in the table above. A state dict is a Python
dictionary that maps each layer in the model to its learned weights and biases.
Unlike the full model (pickle) approach, saving a state dict does not serialize
the model architecture — only the weights are written to disk. This means the
model class or architecture definition must be available when the weights are
loaded back. The tradeoff is worthwhile: state dict files are smaller, safer,
and more portable than pickle-based saves.

The example below trains a simple feedforward regression network on the classic
Auto MPG dataset, which predicts a vehicle's fuel efficiency from attributes
such as engine displacement, horsepower, and weight. After training, the model's
state dict is saved to a `.pth` file using `torch.save`. The weights are then
loaded back into a freshly constructed model using `torch.load` with the
`weights_only=True` argument, which prevents pickle from deserializing arbitrary
objects and should always be specified when loading state dicts in PyTorch 2.0
and later. The RMSE score is printed both before the save and after the load to
confirm that the round-trip preserved the weights exactly.

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# For reproducibility
torch.manual_seed(0)
np.random.seed(0)

# Read the MPG dataset
df = pd.read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv",
    na_values=["NA", "?"]
)

# Handle missing value
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

# Select features and target
features = df[[
    "cylinders", "displacement", "horsepower",
    "weight", "acceleration", "year", "origin"
]]
target = df["mpg"]

# Normalize the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert to PyTorch tensors
features_tensor = torch.tensor(scaled_features, device=device, dtype=torch.float32)
target_tensor = torch.tensor(target.values, device=device, dtype=torch.float32)

# Create DataLoader
dataset = TensorDataset(features_tensor, target_tensor)
data_loader = DataLoader(dataset, batch_size=32)

# Define the model
model = nn.Sequential(
    nn.Linear(features_tensor.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1),
).to(device)

# Loss function and optimizer
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
model.train()
for epoch in range(1000):
    for batch_features, batch_target in data_loader:
        optimizer.zero_grad()
        out = model(batch_features).flatten()
        loss = loss_fn(out, batch_target)
        loss.backward()
        optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, loss: {loss.item():.4f}")

# Evaluate before save
model.eval()
with torch.no_grad():
    pred = model(features_tensor)
score = torch.sqrt(nn.functional.mse_loss(pred.flatten(), target_tensor))
print(f"Before save score (RMSE): {score:.4f}")

# Save using state dict (recommended approach)
torch.save(model.state_dict(), "mpg.pth")

# Reconstruct the model architecture and load saved weights
loaded_model = nn.Sequential(
    nn.Linear(features_tensor.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1),
).to(device)

loaded_model.load_state_dict(torch.load("mpg.pth", weights_only=True))
loaded_model.eval()

# Evaluate after load
with torch.no_grad():
    pred = loaded_model(features_tensor)
score = torch.sqrt(nn.functional.mse_loss(pred.flatten(), target_tensor))
print(f"After load score (RMSE): {score:.4f}")

Epoch 0, loss: 743.8233
Epoch 100, loss: 13.6903
Epoch 200, loss: 10.8130
Epoch 300, loss: 6.9541
Epoch 400, loss: 6.1118
Epoch 500, loss: 5.8200
Epoch 600, loss: 4.4230
Epoch 700, loss: 3.7938
Epoch 800, loss: 4.1214
Epoch 900, loss: 4.4578
Before save score (RMSE): 1.6319
After load score (RMSE): 1.6319


The example below demonstrates a critical characteristic of the state dict
format: saving a model's state dict records only the learned weights, not the
network architecture. To reload the weights, the architecture must be explicitly
redefined in code before calling `load_state_dict`. This is not a limitation so
much as a deliberate design choice — it enforces a clean separation between
structure and parameters, which becomes important in more advanced workflows
such as transfer learning and fine-tuning where weights from one model are
loaded into a modified architecture.

Notice that the network definition in the cell below is identical to the one
used during training. PyTorch's `load_state_dict` works by matching the keys in
the saved dictionary to the layers in the model by name. If the architecture
does not match — for example, if a layer is added, removed, or resized —
PyTorch will raise an error because the keys or tensor shapes will not align.
This strict matching behavior is actually useful: it means a corrupted or
mismatched save will fail loudly rather than silently producing incorrect
results.

After loading, the RMSE score printed below should be identical to the score
printed at the end of the training cell, confirming that the round-trip
preserved the weights exactly.

In [3]:
# Load the saved state dict into a fresh model instance
loaded_model = nn.Sequential(
    nn.Linear(features_tensor.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1),
).to(device)

loaded_model.load_state_dict(torch.load("mpg.pth", weights_only=True))
loaded_model.eval()

# Evaluate the loaded model — RMSE should match the pre-save score
with torch.no_grad():
    pred = loaded_model(features_tensor)

score = torch.sqrt(nn.functional.mse_loss(pred.flatten(), target_tensor))
print(f"After load score (RMSE): {score:.4f}")

After load score (RMSE): 1.6319


# Module 4 Assignment

You can find the second assignment here: [assignment 4](https://github.com/jeffheaton/app_deep_learning/blob/main/assignments/assignment_yourname_t81_558_class4.ipynb)
